# Equity Duration Notebook

Ziel: Konstruktion mehrerer Equity-Duration-Maße für STOXX Europe 600 Unternehmen, jeweils streng as-of Year-End.

Die Pipeline ist in 7 klaren Schritten organisiert:

1. Setup, Pfade, Session
2. Universum laden (STOXX 600 EURO-HQ)
3. Helper-Funktionen (Pulls + Finance-Math)
4. Daily Prices → Daily Returns (Basisdaten)
5. Firm-level Beta und CAPM Cost of Equity
6. Analyst CFPS Forecasts (FY0–FY4)
7. Year-end Prices, 
8. CFPS Duration-Maße
9. Empirische Duration
10. Final Table für weitere Analyse

Hinweis: CFPSMean_* ist hier operating cash flow per share (CFO/share), nicht Net-Payout.


## Step 1 — Setup, Imports, Pfade, Session

Was hier passiert:
- Bibliotheken importieren
- Projektpfade und I/O-Konventionen setzen
- LSEG-Session vorbereiten (falls nötig)


In [2]:
import re
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path
import time
import random

import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="lseg"
)

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)


## Step 2 — Universum laden: STOXX 600 (EURO subset)

Was hier passiert:
- Constituents aus der Membership-Matrix laden
- Spalten harmonisieren (z. B. RIC, Name, CountryHQ, MarketCap)
- Ein sauberes Firmen-Universum für alle folgenden Pulls erzeugen


In [3]:
# --- User input ---
INPUT_XLSX = TABLE_DIR / "stoxx600_membership_matrix_1999_2025_eurohq.xlsx"
SHEET_NAME = 0  # or a sheet name string

df_const = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Robust column detection
ric_col = next(c for c in df_const.columns if "RIC" in c.upper())
name_col = next(c for c in df_const.columns if "COMPANY" in c.upper() or "NAME" in c.upper())
country_col = next(c for c in df_const.columns if "COUNTRY" in c.upper())

df_basic = df_const[[ric_col, name_col, country_col]].copy()
df_basic.columns = ["RIC", "CompanyName", "CountryHQ"]

df_basic = (
    df_basic
    .dropna(subset=["RIC"])
    .drop_duplicates(subset=["RIC"])
    .sort_values("RIC")
    .reset_index(drop=True)
)

# print(df_basic.head())
print("Number of firms:", len(df_basic))


Number of firms: 667


## Step 3 — Helper-Funktionen: Datenpull und Finance-Math

Was hier passiert:
- Kleine, wiederverwendbare Funktionen definieren
- Typische Bausteine: Year-end Price, Beta, CAPM, Duration-Formeln
- Ziel: Logik kapseln, später weniger Copy/Paste


In [4]:

def get_beta_5y(ric: str) -> float:
    """5Y beta from LSEG."""
    beta_df = ld.get_data(universe=[ric], fields=["TR.BetaFiveYear"])
    if beta_df is None or beta_df.empty:
        return np.nan

    beta_col = next((c for c in beta_df.columns if "Beta" in c), None)
    if beta_col is None:
        beta_col = beta_df.columns[-1]

    b = beta_df[beta_col].iloc[0]
    return float(b) if b is not None else np.nan


def coe_capm(beta: float, rf: float, erp: float) -> float:
    """CAPM cost of equity in decimals."""
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return np.nan
    return rf + beta * erp


def duration_discounted(CF: np.ndarray, P: float, r: float) -> float:
    """Implied duration with terminal value inferred from price."""
    if not np.isfinite(P) or P <= 0 or not np.isfinite(r) or r <= 0:
        return np.nan
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan

    t = np.arange(1, T + 1, dtype=float)
    pv_cf = CF / (1.0 + r) ** t
    pv_sum = pv_cf.sum()

    term1 = (t * pv_cf).sum() / P
    term2 = (T + (1.0 + r) / r) * ((P - pv_sum) / P)
    return float(term1 + term2)


def duration_undiscounted(CF: np.ndarray) -> float:
    """Cash-flow timing duration (undiscounted)."""
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan
    t = np.arange(1, T + 1, dtype=float)
    return float((t * CF).sum() / CF.sum()) if CF.sum() != 0 else np.nan


## Step 4 — Daily Prices ziehen und Daily Returns bauen

Was hier passiert:
- Daily Preise für alle RICs ziehen
- Simple Returns berechnen
- Ergebnis als zentrales Input-Panel rets_daily speichern/laden


In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def chunk_list(x, n=30):
    for i in range(0, len(x), n):
        yield x[i:i+n]

def make_date_blocks(start: str, end: str, mode: str = "5y"):
    """
    mode: "yearly" or "5y"
    returns list of (start, end) as YYYY-MM-DD strings (inclusive endpoints for API calls)
    """
    start_dt = pd.Timestamp(start)
    end_dt   = pd.Timestamp(end)

    blocks = []
    cur = start_dt

    if mode not in {"yearly", "5y"}:
        raise ValueError("mode must be 'yearly' or '5y'")

    step_years = 1 if mode == "yearly" else 5

    while cur <= end_dt:
        nxt = cur + pd.DateOffset(years=step_years) - pd.DateOffset(days=1)
        blk_end = min(nxt, end_dt)
        blocks.append((cur.strftime("%Y-%m-%d"), blk_end.strftime("%Y-%m-%d")))
        cur = blk_end + pd.DateOffset(days=1)

    return blocks

def _safe_get_history(batch, field, start, end, interval="daily",
                      max_retries=5, base_sleep=0.8):
    """
    Robust wrapper around ld.get_history with retry/backoff.
    Returns df or None.
    """
    last_err = None
    for r in range(max_retries):
        try:
            df = ld.get_history(
                universe=batch,
                fields=[field],
                start=start,
                end=end,
                interval=interval
            )
            if df is not None and not df.empty:
                return df
        except Exception as e:
            last_err = e

        # exponential backoff + jitter
        sleep_s = base_sleep * (2 ** r) + random.random() * 0.5
        time.sleep(sleep_s)

    if last_err is not None:
        print(f"[WARN] Batch failed after retries. ({start}..{end}) Example RIC: {batch[0]} | {last_err}")
    else:
        print(f"[WARN] Batch returned empty after retries. ({start}..{end}) Example RIC: {batch[0]}")
    return None

def _ensure_wide_with_date(df, batch):
    """
    Tries to convert the returned object into a DataFrame with:
    columns: ["date"] + batch_rics_present
    """
    dfw = df.copy()

    # ensure date column
    if isinstance(dfw.index, pd.DatetimeIndex):
        dfw = dfw.reset_index()
        # name could be None or something else
        if "date" not in [c.lower() for c in dfw.columns]:
            dfw = dfw.rename(columns={dfw.columns[0]: "date"})
        else:
            # if there is already a date-like column name, unify it to "date"
            date_col = next((c for c in dfw.columns if str(c).lower() == "date"), dfw.columns[0])
            if date_col != "date":
                dfw = dfw.rename(columns={date_col: "date"})
    else:
        dfw = dfw.reset_index()
        date_col = next((c for c in dfw.columns if "date" in str(c).lower()), None)
        if date_col is None:
            date_col = dfw.columns[0]
        if date_col != "date":
            dfw = dfw.rename(columns={date_col: "date"})

    dfw["date"] = pd.to_datetime(dfw["date"])

    # keep only date + columns matching rics in batch (intersection)
    keep_cols = ["date"] + [c for c in dfw.columns if c in set(batch)]
    dfw = dfw[keep_cols].copy()

    return dfw

def _wide_to_long(dfw, value_name="value"):
    """
    Converts wide (date + RIC columns) to long (date, RIC, value).
    """
    long = dfw.melt(id_vars=["date"], var_name="RIC", value_name=value_name)
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    long = long.dropna(subset=["value"])
    return long

def infer_totalreturn_is_level(rets_long: pd.DataFrame, sample_rics=8):
    """
    Heuristic:
    - If typical absolute values are small (e.g., around 0.0xx), it's likely already returns.
    - If values are large and positive (e.g., 80, 100, 200) and mostly > 0, it's likely an index/level.
    We test on a sample of RICs.
    """
    if rets_long.empty:
        return False

    sample = (rets_long
              .sort_values(["RIC", "date"])
              .groupby("RIC", as_index=False)
              .head(50))

    # reduce to few RICs
    rics = sample["RIC"].dropna().unique()[:sample_rics]
    sample = sample[sample["RIC"].isin(rics)]

    vals = sample["value"].dropna().values
    if vals.size == 0:
        return False

    p50 = np.nanmedian(vals)
    p90 = np.nanpercentile(vals, 90)

    # Returns typically in [-0.2, 0.2] daily; levels typically much larger.
    # This is a heuristic; you can tighten/loosen thresholds.
    looks_like_level = (p90 > 5) and (p50 > 1)  # e.g., 100-ish index
    return bool(looks_like_level)

# -----------------------------
# Main pull: TR.TotalReturn in time blocks + batches
# -----------------------------
def get_total_return_daily(
    rics,
    start="1999-01-01",
    end=None,
    field="TR.TotalReturn",
    batch_size=30,
    block_mode="5y",              # "yearly" or "5y"
    checkpoint_dir=None,          # e.g. Path(".../intermediate/total_return_blocks")
    max_retries=5
):
    """
    Pull TR.TotalReturn in (time blocks x RIC batches), with retries and optional checkpoints.
    Returns long: date, RIC, value (raw from API)
    """
    # default end = today
    if end is None:
        end = pd.Timestamp.today().strftime("%Y-%m-%d")

    # clean RICs
    rics = [r for r in pd.Series(rics).dropna().astype(str).unique().tolist() if r.strip() != ""]
    blocks = make_date_blocks(start, end, mode=block_mode)

    if checkpoint_dir is not None:
        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

    out = []

    for blk_ix, (blk_start, blk_end) in enumerate(blocks, start=1):
        for b_ix, batch in enumerate(chunk_list(rics, n=batch_size), start=1):

            ck_path = None
            if checkpoint_dir is not None:
                ck_path = checkpoint_dir / f"blk_{blk_ix:03d}_{blk_start}_{blk_end}__batch_{b_ix:03d}.parquet"
                if ck_path.exists():
                    out.append(pd.read_parquet(ck_path))
                    continue

            df = _safe_get_history(
                batch=batch,
                field=field,
                start=blk_start,
                end=blk_end,
                interval="daily",
                max_retries=max_retries
            )
            if df is None or df.empty:
                continue

            dfw = _ensure_wide_with_date(df, batch)
            if dfw.shape[1] <= 1:
                continue

            df_long = _wide_to_long(dfw, value_name="value")

            if ck_path is not None:
                df_long.to_parquet(ck_path, index=False)

            out.append(df_long)

        print(f"[INFO] Finished block {blk_ix}/{len(blocks)}: {blk_start}..{blk_end}")

    if not out:
        return pd.DataFrame(columns=["date", "RIC", "value"])

    res = pd.concat(out, ignore_index=True)
    res["date"] = pd.to_datetime(res["date"])
    return res

def to_daily_returns_from_totalreturn(raw_long: pd.DataFrame):
    """
    Converts raw TR.TotalReturn output to daily returns if it looks like a level/index.
    If it already looks like returns, keeps as-is.

    Output: date, RIC, ret
    """
    if raw_long.empty:
        return pd.DataFrame(columns=["date", "RIC", "ret"])

    raw_long = raw_long.sort_values(["RIC", "date"]).copy()

    is_level = infer_totalreturn_is_level(raw_long)
    if is_level:
        # Treat as level/index -> convert to returns
        raw_long["ret"] = raw_long.groupby("RIC")["value"].pct_change()
        out = raw_long.dropna(subset=["ret"])[["date", "RIC", "ret"]].copy()
        print("[INFO] TR.TotalReturn looked like a LEVEL/INDEX. Converted via pct_change().")
        return out
    else:
        # Treat as already a return series
        out = raw_long.rename(columns={"value": "ret"})[["date", "RIC", "ret"]].copy()
        print("[INFO] TR.TotalReturn looked like RETURNS already. Kept as-is.")
        return out

# -----------------------------
# Example usage
# -----------------------------
# Optional: choose a checkpoint folder (highly recommended for big pulls)
# checkpoint_dir = DATA_DIR / "tr_totalreturn_checkpoints"
checkpoint_dir = None

ld.open_session()

raw_tr = get_total_return_daily(
    rics=df_basic["RIC"].tolist(),
    start="1999-01-01",
    end=None,                  # <-- (2) automatically today
    field="TR.TotalReturn",
    batch_size=30,
    block_mode="5y",           # <-- (3) "yearly" or "5y"
    checkpoint_dir=checkpoint_dir,
    max_retries=5
)

ld.close_session()

rets_daily = to_daily_returns_from_totalreturn(raw_tr)
rets_daily = rets_daily.sort_values(["RIC", "date"]).reset_index(drop=True)

rets_daily.head()

In [ ]:
# --- Save daily returns (Parquet) ---
save_parquet(rets_daily, "returns_daily")

In [5]:
# --- Load daily returns (Parquet) ---
RET_FILE = DATA_DIR / "returns_daily.parquet"
rets_daily = pd.read_parquet(RET_FILE)

## Step 5 — Firm-level Beta und CAPM Cost of Equity

Was hier passiert:
- Marktserie definieren (z. B. .STOXX)
- Rolling/Window-Betas pro Firma schätzen (5Y, daily)
- Daraus CAPM-Kapitalkosten r_i ableiten

Output (typisch):
- beta_panel (RIC × Year)
- COE_CAPM als Input fürs Discounting


### Hintergrund: CAPM-Beta (kurz)

Definition:
$$
\beta_i = \frac{\mathrm{Cov}(r_{i,t}, r_{m,t})}{\mathrm{Var}(r_{m,t})}
$$

Intuition:
- Beta ist die Sensitivität der Aktie gegenüber dem Markt
- Empirisch: Steigung aus der Regression r_{i,t} auf r_{m,t}
- Hier: Beta wird mit 5-Jahres-Fenstern aus Daily Returns geschätzt


In [ ]:
# ============================================================
# Inputs
# ============================================================
# Your daily returns table (long): date | RIC | ret
# - ret seems to be in PERCENT units (e.g., -1.59 means -1.59%)
# - we'll convert to DECIMAL units consistently

RF  = 0.025  # risk-free rate (decimal)
ERP = 0.05   # equity risk premium (decimal)

# Choose your market index RIC (Refinitiv)
MKT_RIC = ".STOXX"          # e.g. STOXX Europe 600 (adjust if needed)
SDate   = "1999-01-01"
EDate   = "2025-12-31"

# Outlier handling (recommended)
RI_CLIP = (-0.50, 0.50)     # clip stock daily returns to [-50%, +50%] after scaling
RM_CLIP = (-0.20, 0.20)     # clip market daily returns to [-20%, +20%] after scaling

MIN_OBS = 400               # minimum daily obs in the 5Y window to compute beta


# ============================================================
# Helper: CAPM cost of equity
# ============================================================
def coe_capm(beta: float, rf: float = RF, erp: float = ERP) -> float:
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return np.nan
    return rf + beta * erp


# ============================================================
# 1) Prepare stock returns from rets_daily
# ============================================================
# Ensure correct dtypes
rets_daily = rets_daily.copy()
rets_daily["date"] = pd.to_datetime(rets_daily["date"], errors="coerce")
rets_daily["ret"]  = pd.to_numeric(rets_daily["ret"], errors="coerce")
rets_daily = rets_daily.dropna(subset=["date", "RIC", "ret"])

# Convert stock returns from percent to decimal
rets_daily["ri"] = rets_daily["ret"] / 100.0

# Optional but highly recommended: clip extreme outliers (data errors, splits, etc.)
rets_daily["ri"] = rets_daily["ri"].clip(RI_CLIP[0], RI_CLIP[1])

# Keep only needed columns
stocks = rets_daily[["date", "RIC", "ri"]].copy()


# ============================================================
# 2) Pull market index price close and compute market returns
#    NOTE: In your environment, you must request Date explicitly via TR.PriceClose.Date
# ============================================================
ld.open_session()

mkt = ld.get_data(
    universe=[MKT_RIC],
    fields=["TR.PriceClose.Date", "TR.PriceClose"],
    parameters={"SDate": SDate, "EDate": EDate, "FRQ": "D"}
)

ld.close_session()

mkt = mkt.rename(columns=lambda c: c.strip())

# Your environment returns columns: Instrument, Date, Price Close
if "Date" not in mkt.columns or "Price Close" not in mkt.columns:
    raise ValueError(f"Unexpected market data shape. Columns: {mkt.columns.tolist()}")

mkt["Date"] = pd.to_datetime(mkt["Date"], errors="coerce")
mkt["Price Close"] = pd.to_numeric(mkt["Price Close"], errors="coerce")

mkt = (
    mkt.dropna(subset=["Date", "Price Close"])
       .sort_values("Date")
       .drop_duplicates(subset=["Date"], keep="last")
       .set_index("Date")["Price Close"]
)

rm = mkt.pct_change().rename("rm").dropna()

# Clip extreme market moves (optional)
rm = rm.clip(RM_CLIP[0], RM_CLIP[1])

# ============================================================
# 3) Merge stocks with market returns (inner join on dates)
# ============================================================
df = (
    stocks.merge(rm, left_on="date", right_index=True, how="inner")
          .dropna(subset=["ri", "rm"])
          .sort_values(["RIC", "date"])
)

# Sanity checks (should look reasonable now)
print("Stock returns (ri) describe:\n", df["ri"].describe())
print("\nMarket returns (rm) describe:\n", df["rm"].describe())
print("\nRatio std(ri)/std(rm):", float(df["ri"].std() / df["rm"].std()))


# ============================================================
# 4) Fast 5Y beta at each year-end (as-of 31.12.y) using Cov/Var
# ============================================================
def beta_5y_yearends_fast(df: pd.DataFrame, min_obs: int = 400) -> pd.DataFrame:
    """
    df columns: date, RIC, ri, rm (decimal returns)
    returns: RIC | Year | AsOf | BETA_5Y | Nobs
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=["RIC", "Year", "AsOf", "BETA_5Y", "Nobs"])

    d = df.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.dropna(subset=["date", "RIC", "ri", "rm"])
    if d.empty:
        return pd.DataFrame(columns=["RIC", "Year", "AsOf", "BETA_5Y", "Nobs"])

    years = np.sort(d["date"].dt.year.unique())
    out = []

    for y in years:
        asof = pd.Timestamp(year=y, month=12, day=31)
        start = asof - pd.DateOffset(years=5)

        w = d[(d["date"] > start) & (d["date"] <= asof)]
        if w.empty:
            continue

        g = w.groupby("RIC", sort=False)

        n = g["ri"].count()
        mean_ri = g["ri"].mean()
        mean_rm = g["rm"].mean()

        # Bring group means to row level
        w = w.join(mean_ri.rename("mean_ri"), on="RIC")
        w = w.join(mean_rm.rename("mean_rm"), on="RIC")

        crm = w["rm"] - w["mean_rm"]
        cri = w["ri"] - w["mean_ri"]

        w["prod"] = crm * cri
        w["crm2"] = crm * crm

        cov = w.groupby("RIC")["prod"].mean()
        var = w.groupby("RIC")["crm2"].mean()

        beta = cov / var
        beta = beta.where(n >= min_obs)

        panel_y = pd.DataFrame({
            "RIC": beta.index,
            "Year": y,
            "AsOf": f"{y}-12-31",
            "BETA_5Y": beta.values,
            "Nobs": n.reindex(beta.index).values
        })

        panel_y = panel_y.dropna(subset=["BETA_5Y"])
        if not panel_y.empty:
            out.append(panel_y)

    if not out:
        return pd.DataFrame(columns=["RIC", "Year", "AsOf", "BETA_5Y", "Nobs"])

    return pd.concat(out, ignore_index=True)


beta_panel = beta_5y_yearends_fast(df, min_obs=MIN_OBS)

# Add CAPM cost of equity (firm-year)
beta_panel["COE_CAPM"] = RF + beta_panel["BETA_5Y"] * ERP

# Quick check of beta distribution
print("\nBeta distribution:\n",
      beta_panel["BETA_5Y"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

beta_panel.head()

Stock returns (ri) describe:
 count    3049712.0
mean      0.000446
std       0.025294
min           -0.5
25%      -0.009828
50%            0.0
75%        0.01029
max            0.5
Name: ri, dtype: Float64

Market returns (rm) describe:
 count    3049712.0
mean      0.000167
std       0.011612
min      -0.114776
25%      -0.005179
50%       0.000595
75%       0.005924
max       0.098669
Name: rm, dtype: Float64

Ratio std(ri)/std(rm): 2.1781385864891374

Beta distribution:
 count     11956.0
mean     0.890942
std      0.410664
min     -0.287892
1%       0.072644
5%       0.230916
50%       0.88062
95%      1.590873
99%       1.89444
max      2.563078
Name: BETA_5Y, dtype: Float64


,RIC,Year,AsOf,BETA_5Y,Nobs,COE_CAPM
0,1U1.DE,2000,2000-12-31,1.263833,503,0.088192
1,A2.MI,2000,2000-12-31,1.263905,507,0.088195
2,AALB.AS,2000,2000-12-31,0.332238,508,0.041612
3,ACBr.AT,2000,2000-12-31,0.480946,503,0.049047
4,ACCP.PA,2000,2000-12-31,0.623166,507,0.056158


## Step 6 — Analyst CFPS Forecasts (FY0–FY4)

Was hier passiert:
- Konsensschätzungen für Operating CFPS as-of Year-End ziehen
- Horizont: FY0 bis FY4
- Ergebnis als firm-year Panel strukturieren (z. B. cfps_panel)


Die CFPS-Panel-Logik ist strikt as-of 31.12.Y:
- Für jedes Jahr Y werden nur Informationen verwendet, die bis 31.12.Y verfügbar sind
- Die Forecasts bilden Timing und Größenordnung erwarteter Cashflows ab
- Diese Größen sind der zentrale Input für die Duration-Konstruktion


In [81]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
ASOF_YEARS = list(range(1999, 2026))     # 1999 ... 2025
ASOF_MMDD = "12-31"

HORIZONS = ["FY0", "FY1", "FY2", "FY3", "FY4"]
FIELDS = [f"TR.CFPSMean(period={h})" for h in HORIZONS]

BATCH_SIZE = 100   # konservativ → stabil
rics = df_basic["RIC"].dropna().astype(str).unique().tolist()


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


# ------------------------------------------------------------
# Pull CFPS forecasts (all RICs, all years)
# ------------------------------------------------------------
def pull_cfps_panel_all_rics(rics, asof_years):
    rows = []

    ld.open_session()
    try:
        for y in asof_years:
            asof = f"{y}-{ASOF_MMDD}"
            print(f"AsOf {asof}")

            for batch in chunks(rics, BATCH_SIZE):
                df = ld.get_data(
                    universe=batch,
                    fields=FIELDS,
                    parameters={
                        "FRQ": "FY",
                        "SDate": asof,
                        "EDate": asof
                    }
                )

                if df is None or df.empty:
                    continue

                # Erwartetes Layout:
                # Col 0: Instrument
                # Col 1..5: CFPSMean FY0..FY4 (alle gleich benannt!)
                if df.shape[1] < 1 + len(HORIZONS):
                    continue

                for i in range(len(df)):
                    ric = str(df.iloc[i, 0])
                    vals = df.iloc[i, 1:1+len(HORIZONS)].tolist()

                    rec = {
                        "RIC": ric,
                        "AsOfYear": y,
                        "AsOfDate": asof
                    }

                    for h, v in zip(HORIZONS, vals):
                        rec[h] = pd.to_numeric(v, errors="coerce")

                    rows.append(rec)

    finally:
        ld.close_session()

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    out["AsOfDate"] = pd.to_datetime(out["AsOfDate"])

    # Deduplicate: last value wins (safest with Refinitiv batches)
    out = (
        out.sort_values(["RIC", "AsOfDate"])
           .drop_duplicates(["RIC", "AsOfYear"], keep="last")
           .reset_index(drop=True)
    )

    return out


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
cfps_panel = pull_cfps_panel_all_rics(rics, ASOF_YEARS)

print(cfps_panel.head())
print("\nRows:", len(cfps_panel))
print("Unique RICs:", cfps_panel["RIC"].nunique())
print("Missing FY0:", cfps_panel["FY0"].isna().sum())
print("Missing FY4:", cfps_panel["FY4"].isna().sum())

save_parquet(cfps_panel, "cfps_forecasts")

AsOf 1999-12-31
AsOf 2000-12-31
AsOf 2001-12-31
AsOf 2002-12-31
AsOf 2003-12-31
AsOf 2004-12-31
AsOf 2005-12-31
AsOf 2006-12-31
AsOf 2007-12-31
AsOf 2008-12-31
AsOf 2009-12-31
AsOf 2010-12-31
AsOf 2011-12-31
AsOf 2012-12-31
AsOf 2013-12-31
AsOf 2014-12-31
AsOf 2015-12-31
AsOf 2016-12-31
AsOf 2017-12-31
AsOf 2018-12-31
AsOf 2019-12-31
AsOf 2020-12-31
AsOf 2021-12-31
AsOf 2022-12-31
AsOf 2023-12-31
AsOf 2024-12-31
AsOf 2025-12-31
            RIC  AsOfYear   AsOfDate  FY0  FY1  FY2  FY3  FY4
0  1COVG.DE^L25      1999 1999-12-31  NaN  NaN  NaN  NaN  NaN
1  1COVG.DE^L25      2000 2000-12-31  NaN  NaN  NaN  NaN  NaN
2  1COVG.DE^L25      2001 2001-12-31  NaN  NaN  NaN  NaN  NaN
3  1COVG.DE^L25      2002 2002-12-31  NaN  NaN  NaN  NaN  NaN
4  1COVG.DE^L25      2003 2003-12-31  NaN  NaN  NaN  NaN  NaN

Rows: 18009
Unique RICs: 667
Missing FY0: 7548
Missing FY4: 12195
Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/cfps_forecasts.parque

## Step 7 — Year-end Prices

Was hier passiert (gesamter Output-Block):
- Year-end Preise (31.12.Y) ziehen und mit Forecasts mergen


In [ ]:
# ============================================================
# Pull year-end prices (Price Close) for ALL RICs at each year-end (31-12)
# Output: price_panel with columns: RIC | AsOfYear | AsOfDate | Price_12_31
# ============================================================

ASOF_YEARS = list(range(1999, 2026))          # 1999 ... 2025
ASOF_MMDD  = "12-31"
BATCH_SIZE = 200

rics = df_basic["RIC"].dropna().astype(str).unique().tolist()

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def _find_price_col(df: pd.DataFrame) -> str | None:
    # Your environment typically names it "Price Close"
    if "Price Close" in df.columns:
        return "Price Close"
    # Fallbacks if LSEG changes labeling
    for c in df.columns:
        if "price close" in str(c).lower():
            return c
    # Rare case: direct TR.PriceClose column name
    if "TR.PriceClose" in df.columns:
        return "TR.PriceClose"
    return None

rows = []

ld.open_session()
try:
    for y in ASOF_YEARS:
        asof = f"{y}-{ASOF_MMDD}"
        print(f"Pulling PriceClose for {asof} ...")

        for batch in chunks(rics, BATCH_SIZE):
            df = ld.get_data(
                universe=batch,
                fields=["TR.PriceClose.Date", "TR.PriceClose"],
                parameters={
                    "SDate": asof,
                    "EDate": asof,
                    "FRQ": "D"
                }
            )

            if df is None or df.empty:
                continue

            df = df.rename(columns=lambda c: str(c).strip())

            # Identifier
            id_col = "Instrument" if "Instrument" in df.columns else df.columns[0]

            # Date column (should exist because we requested TR.PriceClose.Date)
            if "Date" not in df.columns:
                # If your environment returns a different label, try to find it
                date_col = next((c for c in df.columns if str(c).lower() == "date"), None)
                if date_col is None:
                    continue
            else:
                date_col = "Date"

            # Price column
            price_col = _find_price_col(df)
            if price_col is None:
                continue

            tmp = df[[id_col, date_col, price_col]].copy()
            tmp = tmp.rename(columns={id_col: "RIC", date_col: "Date", price_col: "Price_12_31"})
            tmp["RIC"] = tmp["RIC"].astype(str)
            tmp["Date"] = pd.to_datetime(tmp["Date"], errors="coerce")
            tmp["Price_12_31"] = pd.to_numeric(tmp["Price_12_31"], errors="coerce")
            tmp = tmp.dropna(subset=["RIC", "Price_12_31"])

            if tmp.empty:
                continue

            tmp["AsOfYear"] = y
            tmp["AsOfDate"] = pd.Timestamp(asof)

            # If duplicates appear (rare), keep last
            tmp = tmp.sort_values(["RIC", "Date"]).drop_duplicates(subset=["RIC"], keep="last")

            rows.append(tmp[["RIC", "AsOfYear", "AsOfDate", "Price_12_31"]])

finally:
    ld.close_session()

price_panel = (
    pd.concat(rows, ignore_index=True)
    if rows else
    pd.DataFrame(columns=["RIC", "AsOfYear", "AsOfDate", "Price_12_31"])
)

# Final dedup (last wins)
if not price_panel.empty:
    price_panel = (
        price_panel.sort_values(["RIC", "AsOfDate"])
                   .drop_duplicates(["RIC", "AsOfYear"], keep="last")
                   .reset_index(drop=True)
    )

print(price_panel.head())
print("Rows:", len(price_panel))
print("Unique RICs:", price_panel["RIC"].nunique())
print("Missing prices:", price_panel["Price_12_31"].isna().sum())

#save
save_parquet(price_panel, "price_year_end_panel")

Pulling PriceClose for 1999-12-31 ...
Pulling PriceClose for 2000-12-31 ...
Pulling PriceClose for 2001-12-31 ...
Pulling PriceClose for 2002-12-31 ...
Pulling PriceClose for 2003-12-31 ...
Pulling PriceClose for 2004-12-31 ...
Pulling PriceClose for 2005-12-31 ...
Pulling PriceClose for 2006-12-31 ...
Pulling PriceClose for 2007-12-31 ...
Pulling PriceClose for 2008-12-31 ...
Pulling PriceClose for 2009-12-31 ...
Pulling PriceClose for 2010-12-31 ...
Pulling PriceClose for 2011-12-31 ...
Pulling PriceClose for 2012-12-31 ...
Pulling PriceClose for 2013-12-31 ...
Pulling PriceClose for 2014-12-31 ...
Pulling PriceClose for 2015-12-31 ...
Pulling PriceClose for 2016-12-31 ...
Pulling PriceClose for 2017-12-31 ...
Pulling PriceClose for 2018-12-31 ...
Pulling PriceClose for 2019-12-31 ...
Pulling PriceClose for 2020-12-31 ...
Pulling PriceClose for 2021-12-31 ...
Pulling PriceClose for 2022-12-31 ...
Pulling PriceClose for 2023-12-31 ...
Pulling PriceClose for 2024-12-31 ...
Pulling Pric

## Step 8 — Duration-Maße aus CFPS

Was hier passiert:
- Drei Timing-/Duration-Maße berechnen:
  - Duration_undiscounted
  - Duration_r125 (fixed r = 12.5%)
  - Duration_CAPM (firm-spezifisches r_i)
- Alle Maße sind konsistent auf per-share Größen aufgebaut


In [91]:
HORIZONS = ["FY0", "FY1", "FY2", "FY3", "FY4"]

# Ensure numeric CF columns
for c in HORIZONS:
    cfps_panel[c] = pd.to_numeric(cfps_panel[c], errors="coerce")

# Helper: build CF array from a row (FY0..FY4)
def row_cf_array(row) -> np.ndarray:
    return row[HORIZONS].to_numpy(dtype=float)

# 1) Undiscounted (cash-flow-timing) duration
cfps_panel["Duration_undiscounted"] = cfps_panel.apply(
    lambda row: duration_undiscounted(row_cf_array(row)),
    axis=1
)

# 2) Merge year-end prices (needed as terminal value P)
#    Assumes you already created: price_panel with columns ["RIC","AsOfYear","Price_12_31"]
cfps_panel = cfps_panel.merge(
    price_panel[["RIC", "AsOfYear", "Price_12_31"]],
    on=["RIC", "AsOfYear"],
    how="left"
)

# 3) Discounted duration at fixed r = 12.5%
R_FIXED = 0.125

def safe_duration_rfixed(row) -> float:
    cf = row_cf_array(row)
    P = row["Price_12_31"]
    if np.all(np.isnan(cf)) or pd.isna(P) or P <= 0:
        return np.nan
    return duration_discounted(cf, P=P, r=R_FIXED)

cfps_panel["Duration_r125"] = cfps_panel.apply(safe_duration_rfixed, axis=1)

cfps_panel[["RIC", "AsOfYear", "Price_12_31", "Duration_undiscounted", "Duration_r125"]].head()

,RIC,AsOfYear,Price_12_31,Duration_undiscounted,Duration_r125
0,1COVG.DE^L25,1999,<NA>,NaN,NaN
1,1COVG.DE^L25,2000,<NA>,NaN,NaN
2,1COVG.DE^L25,2001,<NA>,NaN,NaN
3,1COVG.DE^L25,2002,<NA>,NaN,NaN
4,1COVG.DE^L25,2003,<NA>,NaN,NaN


In [92]:
# Assumes you already have:
# - cfps_panel with FY0..FY4, Price_12_31, Duration_undiscounted, Duration_r125
# - beta_panel (or similar) containing firm-year COE_CAPM (your r_i,y from CAPM)
# - duration_discounted(cf, P, r) defined

HORIZONS = ["FY0", "FY1", "FY2", "FY3", "FY4"]

def row_cf_array(row) -> np.ndarray:
    return row[HORIZONS].to_numpy(dtype=float)

# 1) Bring in firm-year CAPM discount rates r_{i,y}
# beta_panel might use "Year" instead of "AsOfYear"
coe = beta_panel.copy()
if "AsOfYear" not in coe.columns and "Year" in coe.columns:
    coe = coe.rename(columns={"Year": "AsOfYear"})

coe = coe[["RIC", "AsOfYear", "COE_CAPM"]].drop_duplicates(["RIC", "AsOfYear"], keep="last")
coe["AsOfYear"] = pd.to_numeric(coe["AsOfYear"], errors="coerce").astype("Int64")
coe["COE_CAPM"] = pd.to_numeric(coe["COE_CAPM"], errors="coerce")

cfps_panel["AsOfYear"] = pd.to_numeric(cfps_panel["AsOfYear"], errors="coerce").astype("Int64")

cfps_panel = cfps_panel.merge(
    coe,
    on=["RIC", "AsOfYear"],
    how="left"
)

# 2) Compute CAPM-discounted duration
def safe_duration_capm(row) -> float:
    cf = row_cf_array(row)
    P  = row.get("Price_12_31", np.nan)
    r  = row.get("COE_CAPM", np.nan)

    if np.all(np.isnan(cf)) or pd.isna(P) or P <= 0 or pd.isna(r):
        return np.nan

    # Optional guardrail: avoid pathological r values
    if r <= -0.50 or r >= 1.00:
        return np.nan

    return duration_discounted(cf, P=P, r=float(r))

cfps_panel["Duration_CAPM"] = cfps_panel.apply(safe_duration_capm, axis=1)

cfps_panel[["RIC", "AsOfYear", "Price_12_31", "COE_CAPM", "Duration_r125", "Duration_CAPM"]].head()

,RIC,AsOfYear,Price_12_31,COE_CAPM,Duration_r125,Duration_CAPM
0,1COVG.DE^L25,1999,<NA>,<NA>,NaN,NaN
1,1COVG.DE^L25,2000,<NA>,<NA>,NaN,NaN
2,1COVG.DE^L25,2001,<NA>,<NA>,NaN,NaN
3,1COVG.DE^L25,2002,<NA>,<NA>,NaN,NaN
4,1COVG.DE^L25,2003,<NA>,<NA>,NaN,NaN


## Step 9 — Empirische Duration: Zins-Sensitivität

Was hier passiert:
- Daily Delta-y (2Y EUR OIS) mit Daily Returns mergen
- Pro Firma eine einfache Sensitivität schätzen:
$$
r_{i,t} = a_i + b_i \Delta y_t + \varepsilon_{i,t}
\quad\Rightarrow\quad
D^{emp}_i = -b_i
$$
- Ergebnis: eine datengetriebene Duration-Approximation

### 2Y OIS Serie ziehen (RIC EUREON2Y=)

Wir verwenden das verfügbare Feld, z. B. TR.MIDPRICE.


In [117]:
ld.open_session()

# 2Y EUR OIS / EONIA-style series
ric_rate = "EUREON2Y="
field_rate = "TR.MIDPRICE"

y_df = ld.get_history(
    universe=[ric_rate],
    fields=[field_rate],
    start="1999-01-01",
    end="2025-12-31",
    interval="daily"
)

ld.close_session()

# Build daily rate changes (in percentage points)
rate_df = y_df.copy().reset_index()
rate_df.columns = ["date", "y"]   # y in percent
rate_df["date"] = pd.to_datetime(rate_df["date"])
rate_df["y"] = pd.to_numeric(rate_df["y"], errors="coerce")
rate_df = rate_df.dropna(subset=["y"]).sort_values("date")
rate_df["dy"] = rate_df["y"].diff()

rates_daily = rate_df[["date","dy"]].dropna()

rates_daily.head()


,date,dy
2699,2009-05-07,-0.01
2700,2009-05-08,-0.043
2701,2009-05-11,0.034
2702,2009-05-12,0.024
2703,2009-05-13,-0.066


In [118]:
df_2yOIS = rates_daily.copy()
save_parquet(df_2yOIS, "rates_2yOIS_daily")

Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/rates_2yOIS_daily.parquet


In [119]:
rets_daily.info()
rets_daily.head()
rets_daily.tail()

# Plausibilitätschecks
rets_daily.groupby("RIC")["date"].agg(["min", "max"]).head()
rets_daily["ret"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3056104 entries, 0 to 3056103
Data columns (total 4 columns):
 #   Column  Dtype         
---  ------  -----         
 0   date    datetime64[ns]
 1   RIC     object        
 2   ret     Float64       
 3   ri      Float64       
dtypes: Float64(2), datetime64[ns](1), object(1)
memory usage: 99.1+ MB


count      3056104.0
mean        0.058189
std         9.218542
min       -98.537683
25%        -0.982857
50%              0.0
75%          1.02898
max      8528.323636
Name: ret, dtype: Float64

In [120]:
# erste 30 RICs auswählen (stabil, alphabetisch)
first_30_rics = (
    rets_daily["RIC"]
    .drop_duplicates()
    .sort_values()
    .head(30)
    .tolist()
)

preview_wide = (
    rets_daily
    .query("RIC in @first_30_rics")
    .pivot_table(
        index="date",
        columns="RIC",
        values="ret",
        aggfunc="last"   # oder "mean" wenn du Duplikate mitteln willst
    )
    .sort_index()
)

preview_wide.head()

RIC,1COVG.DE^L25,1U1.DE,A2.MI,A3M.MC,AALB.AS,ABGek.MC^I22,ABI.BR,ABNd.AS,ABVX.PA,ACBr.AT,...,ADYEN.AS,AEGN.AS,AENA.MC,AFXG.DE,AG1G.DE,AGES.BR,AGFB.BR,AGL.MI^G23,AGS.MC^F10,AIBG.I
date,,,,,,,,,,,,,,,,,,,,,
1999-01-04,<NA>,-2.628206,14.736447,<NA>,-0.037897,<NA>,<NA>,<NA>,<NA>,7.997947,...,<NA>,3.352661,<NA>,<NA>,<NA>,7.168155,<NA>,2.568634,10.515333,4.222933
1999-01-05,<NA>,0.862069,-1.461109,<NA>,1.590909,<NA>,<NA>,<NA>,<NA>,0.136261,...,<NA>,1.710587,<NA>,<NA>,<NA>,4.081633,<NA>,0.670184,4.280645,3.580402
1999-01-06,<NA>,12.820513,<NA>,<NA>,1.565996,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,0.863636,<NA>,<NA>,<NA>,-1.359477,<NA>,<NA>,<NA>,5.518496
1999-01-07,<NA>,16.666667,-11.993022,<NA>,0.660793,<NA>,<NA>,<NA>,<NA>,3.006329,...,<NA>,-5.588103,<NA>,<NA>,<NA>,-2.06732,<NA>,-1.983003,-4.885496,-2.298851
1999-01-08,<NA>,-1.623377,-0.891972,<NA>,-0.656455,<NA>,<NA>,<NA>,<NA>,0.460829,...,<NA>,-0.71599,<NA>,<NA>,<NA>,-2.841678,<NA>,0.794798,-4.012841,0.117647


### Firm-level Rate Betas und empirische Duration schätzen


In [ ]:
import numpy as np
import pandas as pd

def estimate_rate_beta_duration(
    rets_daily: pd.DataFrame,
    rates_daily: pd.DataFrame,
    min_obs: int = 200,
    agg_ret: str = "last",   # how to resolve duplicate (date,RIC) returns
) -> pd.DataFrame:
    """
    Estimate: r_{i,t} = a_i + b_i * dy_t
    Returns:
      - beta_rate_2yOIS = b_i
      - D_emp_2yOIS     = -b_i
      - n_obs           = number of daily observations used

    Notes:
    - Handles duplicate (date, RIC) rows in rets_daily by aggregating.
    - Keeps only dates where both stock returns and dy are observed.
    - Does NOT change units: make sure ret and dy are in compatible units for interpretation.
      (If ret is in percent and dy is in percentage points, b is "percent per pp".)
    """

    # --- 0) Defensive copies + types ---
    r = rets_daily.copy()
    z = rates_daily.copy()

    r["date"] = pd.to_datetime(r["date"], errors="coerce")
    z["date"] = pd.to_datetime(z["date"], errors="coerce")

    r["RIC"] = r["RIC"].astype(str)
    r["ret"] = pd.to_numeric(r["ret"], errors="coerce")
    z["dy"]  = pd.to_numeric(z["dy"], errors="coerce")

    r = r.dropna(subset=["date", "RIC", "ret"])
    z = z.dropna(subset=["date", "dy"])

    # --- 1) Resolve duplicates: one return per (date, RIC) ---
    # pivot() fails if duplicates exist; we handle it explicitly here.
    if agg_ret == "last":
        r = (
            r.sort_values(["RIC", "date"])
             .drop_duplicates(subset=["RIC", "date"], keep="last")
        )
    elif agg_ret == "mean":
        r = (
            r.groupby(["RIC", "date"], as_index=False)["ret"]
             .mean()
        )
    else:
        raise ValueError("agg_ret must be 'last' or 'mean'.")

    # --- 2) Merge with rates and drop missing ---
    df = (
        r.merge(z[["date", "dy"]], on="date", how="inner")
         .dropna(subset=["ret", "dy"])
         .copy()
    )

    # --- 3) OLS per RIC: y = a + b x ---
    out = []
    for ric, g in df.groupby("RIC", sort=False):
        if len(g) < min_obs:
            continue

        x = g["dy"].to_numpy(dtype=float)
        y = g["ret"].to_numpy(dtype=float)

        # Drop any remaining non-finite entries
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]
        y = y[m]
        if len(y) < min_obs:
            continue

        X = np.column_stack([np.ones_like(x), x])
        alpha, b = np.linalg.lstsq(X, y, rcond=None)[0]

        out.append({
            "RIC": ric,
            "beta_rate_2yOIS": float(b),
            "D_emp_2yOIS": float(-b),
            "n_obs": int(len(y)),
        })

    return pd.DataFrame(out)


# -----------------------------
# Run + merge into df_basic
# -----------------------------
dur_emp = estimate_rate_beta_duration(rets_daily, rates_daily, min_obs=200, agg_ret="last")

df_basic = df_basic.merge(dur_emp, on="RIC", how="left")

print(df_basic[["RIC", "D_emp_2yOIS", "beta_rate_2yOIS", "n_obs"]].head())
print(df_basic["D_emp_2yOIS"].describe())

# Spearman correlation (requires both columns present)
cols = [c for c in ["Duration_r125", "D_emp_2yOIS"] if c in df_basic.columns]
if len(cols) == 2:
    print(df_basic[cols].corr(method="spearman"))
else:
    print("Duration_r125 not found in df_basic (compute/merge it first).")

print("Share with D_emp:", df_basic["D_emp_2yOIS"].notna().mean())

## Step 10 — Net-Payout-basierte Duration (noch nicht gemacht)


## Step 11 — Final Results Table (gerundet und exportierbar)

Was hier passiert:
- Relevante firm-year Outputs zusammenführen
- Spalten benennen und aufräumen
- Ergebnis als finale Analyse-/Export-Tabelle speichern

In [123]:
# ============================================================
# Build a firm-year table: RIC | YEAR | (all durations)
# - Time-varying durations come from cfps_panel (already has AsOfYear)
# - Time-invariant empirical duration (D_emp_2yOIS) is repeated for every year per RIC
# ============================================================

# 1) Start from your firm-year duration panel (from cfps_panel)
#    Make sure these columns exist in cfps_panel:
#    - RIC, AsOfYear (or Year), Duration_undiscounted, Duration_r125, Duration_CAPM
dur_cols = [c for c in ["Duration_undiscounted", "Duration_r125", "Duration_CAPM"] if c in cfps_panel.columns]

firm_year = cfps_panel[["RIC", "AsOfYear"] + dur_cols].copy()
firm_year = firm_year.rename(columns={"AsOfYear": "YEAR"})
firm_year["YEAR"] = pd.to_numeric(firm_year["YEAR"], errors="coerce").astype("Int64")

# 2) Add the time-invariant empirical duration (D_emp_2yOIS) to every firm-year row
#    This comes from your dur_emp output or from df_basic after merging.
#    Prefer the compact source:
if "dur_emp" in globals() and dur_emp is not None and not dur_emp.empty:
    emp = dur_emp[["RIC", "D_emp_2yOIS"]].copy()
else:
    # fallback: if you only have it inside df_basic
    emp = df_basic[["RIC", "D_emp_2yOIS"]].drop_duplicates("RIC").copy()

firm_year = firm_year.merge(emp, on="RIC", how="left")

# 3) Optional: sort nicely
firm_year = firm_year.sort_values(["RIC", "YEAR"]).reset_index(drop=True)

save_parquet(firm_year, "durations_panel")
firm_year.head()

Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/durations_panel.parquet


,RIC,YEAR,Duration_undiscounted,Duration_r125,Duration_CAPM,D_emp_2yOIS
0,1COVG.DE^L25,1999,NaN,NaN,NaN,-1.906057
1,1COVG.DE^L25,2000,NaN,NaN,NaN,-1.906057
2,1COVG.DE^L25,2001,NaN,NaN,NaN,-1.906057
3,1COVG.DE^L25,2002,NaN,NaN,NaN,-1.906057
4,1COVG.DE^L25,2003,NaN,NaN,NaN,-1.906057
